In [1]:
import os
os.environ['HF_HOME']=r'F:\Langchain\models'

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [4]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [5]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [6]:
multiply.name

'multiply'

In [7]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [8]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

# tool binding

In [9]:
llm=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=100
)

model =ChatHuggingFace(llm=llm)


In [10]:
llm = model

In [11]:
llm.invoke('hi')

AIMessage(content='Hi! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 11, 'total_tokens': 21}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a02e2e-51a3-7661-bfc9-5ea760816a33-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 10, 'total_tokens': 21})

In [12]:
llm_with_tools = llm.bind_tools([multiply])

In [13]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 2, "b": 3}', 'name': 'multiply', 'description': None}, 'id': 'call_HiN2ofFNuTgQPC70obzyZNUt', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 234, 'total_tokens': 300}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a02e2e-56ba-7873-a5a4-19ca026e4f12-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'call_HiN2ofFNuTgQPC70obzyZNUt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 234, 'output_tokens': 66, 'total_tokens': 300})

In [14]:
query = HumanMessage('can you multiply 3 with 1000')

In [15]:
messages = [query]

In [16]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [17]:
result = llm_with_tools.invoke(messages)

In [18]:
result

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'call_HhqTan4KnMQ0fqk3T63hUBGF', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 239, 'total_tokens': 261}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a02e2e-6242-7b51-bac8-fa10a13aa9df-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_HhqTan4KnMQ0fqk3T63hUBGF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 239, 'output_tokens': 22, 'total_tokens': 261})

In [19]:
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 1000},
  'id': 'call_HhqTan4KnMQ0fqk3T63hUBGF',
  'type': 'tool_call'}]

In [20]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 1000},
 'id': 'call_HhqTan4KnMQ0fqk3T63hUBGF',
 'type': 'tool_call'}

In [21]:
messages.append(result)

In [22]:
tool_result = multiply.invoke(result.tool_calls[0])

In [23]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='call_HhqTan4KnMQ0fqk3T63hUBGF')

In [24]:
messages.append(tool_result)

In [25]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'call_HhqTan4KnMQ0fqk3T63hUBGF', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 239, 'total_tokens': 261}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a02e2e-6242-7b51-bac8-fa10a13aa9df-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_HhqTan4KnMQ0fqk3T63hUBGF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 239, 'output_tokens': 22, 'total_tokens': 261}),
 ToolMessage(content='3000', name='multiply', tool_call_id='call_HhqTan4KnMQ0fqk3T63hUBGF')]

In [26]:
llm_with_tools.invoke(messages)

AIMessage(content='The product of 3 and 1000 is indeed 3000.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 271, 'total_tokens': 287}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a02e2e-6697-7460-8d3a-6845cc569cc4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 271, 'output_tokens': 16, 'total_tokens': 287})

In [27]:
llm_with_tools.invoke(messages).content

''

In [41]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/8ffd1b0484c2971e3d389c29/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [42]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [43]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

# tool binding


In [44]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [45]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [46]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [47]:
ai_message = llm_with_tools.invoke(messages)

In [48]:
messages.append(ai_message)

In [49]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR',
   'target_currency': 'USD',
   'conversion_factor': '0.0141'},
  'id': 'call_aerXeQ0D9AePwQOEEEhNOtLS',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'target_currency_value': '0.141'},
  'id': 'call_uGu8GtNjxQre212NO5ksoTVX',
  'type': 'tool_call'}]

In [50]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [51]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.0141, and based on that, 10 INR is equal to approximately 0.1044 USD.'

### agent

In [56]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    Get the currency conversion rate.

    Arguments:
    base_currency: Three-letter currency code such as USD.
    target_currency: Three-letter currency code such as INR.

    Return only the numeric conversion rate.
    """

    url = f'https://v6.exchangerate-api.com/v6/89d1b04e3d3ffc29719c2884/pair/{base_currency}/{target_currency}'

    response = requests.get(url)
    data = response.json()

    return data["conversion_rate"]

In [57]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[get_conversion_factor],
    system_prompt="""
You are a helpful currency assistant.

When the user asks for a currency conversion:
1. Identify the base currency.
2. Identify the target currency.
3. Call get_conversion_factor.
4. Pass ONLY these two arguments:
   base_currency
   target_currency
5. Do not add any other arguments.
6. The tool arguments must be a JSON object, not a list.
"""
)


In [58]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "hi"}
    ]
})

print(response["messages"][-1].content)

What can I help you with today?


In [59]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Convert 10 USD to INR"
        }
    ]
})

print(response["messages"][-1].content)

You wanted to know the conversion rate, and I provided it. If you want the conversion of 10 USD to INR, you can multiply 10 by the conversion rate.

10 * 95.743 = 957.43
